<div class="alert alert-success"><h1>Land cover classification</h1></div>

In [12]:
import ee
import geemap
import os
import pandas as pd

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project = "ee-leviekytz")

In [4]:
import geemap

### Working with GEE
  
- Obtaining the mombasa shapefile and visualizing the shapefile

In [5]:
Map = geemap.Map() 
gaul = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")
mombasa = gaul.filter(ee.Filter.eq("ADM2_NAME","Mombasa"))
roi = mombasa.geometry()
Map.centerObject(mombasa, 11)
Map.addLayer(mombasa,{},'Mombasa')
Map

Map(center=[-4.018361963568997, 39.65151533372141], controls=(WidgetControl(options=['position', 'transparent_…

- Loading the sentinel 2 imagery

In [6]:
sentinel = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
s2 = (sentinel.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',10))
      .filterBounds(roi)
      .filterDate('2025-01-01', '2025-12-31'))
s2_clean = s2.median().clip(roi)

visparams = {
    'bands': ['B4','B3','B2'],
    'min': 0,
    'max': 3000
}

Map.addLayer(s2_clean,visparams,'Mombasa sentinel')
Map


Map(bottom=268301.0, center=[-4.018361963568997, 39.65151533372141], controls=(WidgetControl(options=['positio…

In [7]:
#Extracting water freatures
water_features = ee.FeatureCollection(Map.draw_features)

#Creating a function to store the different landc covers
def assign_water_class(feature):
    return feature.set('landcover', 0)

#Apply the class to the drawing feature
water_training = water_features.map(assign_water_class)
print("water_training_data captured!")

water_training_data captured!


In [8]:
#Extracting built-up areas
builtup_features = ee.FeatureCollection(Map.draw_features)

#Creating a function for built-up areas
def assign_builtup_class(feature):
    return feature.set('landcover', 1)

#Applying the class to the drawing feature
builtup_training=  builtup_features.map(assign_builtup_class)
print("Built up training data captured")

Built up training data captured


In [9]:
#Extracting vegetation
vegetation_features = ee.FeatureCollection(Map.draw_features)

#Creating a function for vegetation
def assign_vegetation_class(feature):
    return feature.set('landcover', 2)

#Assign the class to the drawing feature
vegetation_training = vegetation_features.map(assign_vegetation_class)
print("vegetation training data captured!")

vegetation training data captured!


In [10]:
#Extracting bareground
bareground_features = ee.FeatureCollection(Map.draw_features)

#Creating a function for baregroung
def assign_bareground_class(feature):
    return feature.set('landcover', 3)

#Assign the class to the drawing feature
bareground_training = bareground_features.map(assign_bareground_class)
print("bareground training data captured!")

bareground training data captured!


In [11]:
#Merging the training data
final_training_data=  water_training.merge(builtup_training).merge(vegetation_training).merge(bareground_training)
print("Total training polygons",final_training_data.size().getInfo())

Total training polygons 67


### Exporting to my local pc

In [ ]:
output_file = "mombasa_training.shp"

#Export the merged feature collection to a shapefile
geemap.ee_export_vector(final_training_data, file_name = output_file)